# 02 Web Log Anomaly Detection - LSTM Autoencoder

Train a dedicated **LSTM Autoencoder** for **web logs**.

In [61]:

import os, json, joblib
import numpy as np
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath(".."))
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from src.preprocessing import clean_column_names, basic_log_preprocess, add_text_length_features, fill_numeric, encode_categoricals, keep_or_create_columns, scale_features, create_sequences

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data/logs/web/web_logs.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'models', 'web_logs')
os.makedirs(MODEL_DIR, exist_ok=True)


## 1) Load dataset

In [62]:
df = pd.read_csv(DATA_PATH)
df.head()

,Unnamed: 0,Method,User-Agent,Pragma,Cache-Control,Accept,Accept-encoding,Accept-charset,language,host,cookie,content-type,connection,lenght,content,classification,URL
0,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=1F767F17239C9B670A39E9B10C3825F4,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/index.jsp HTTP/1.1
1,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/publico/anadir.j...
2,Normal,POST,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=933185092E0B668B90676E0A2B0767AF,application/x-www-form-urlencoded,Connection: close,Content-Length: 68,id=3&nombre=Vino+Rioja&precio=100&cantidad=55&...,0,http://localhost:8080/tienda1/publico/anadir.j...
3,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=8FA18BA82C5336D03D3A8AFA3E68CBB0,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/publico/autentic...
4,Normal,POST,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=7104E6C68A6BCF1423DAE990CE49FEE2,application/x-www-form-urlencoded,Connection: close,Content-Length: 63,modo=entrar&login=choong&pwd=d1se3ci%F3n&remem...,0,http://localhost:8080/tienda1/publico/autentic...


## 2) Configure columns
عدل القوائم دي حسب الداتا عندك.

In [63]:

timestamp_col = 'timestamp'
text_cols = ['method', 'url', 'path', 'query', 'user_agent']
feature_candidates = ['status_code', 'response_time', 'request_size', 'response_size', 'hour', 'dayofweek']
label_col ='classification'
sequence_length = 10
lstm_units = 128
latent_dim = 32
dropout_rate = 0.2
learning_rate = 1e-3
batch_size = 32
epochs = 20
threshold_percentile = 65
train_ratio = 0.7

In [64]:
def create_sequences(data, sequence_length=10):
    sequences = []
    for i in range(len(data) - sequence_length + 1):
        sequences.append(data[i:i + sequence_length])
    return np.array(sequences)

## 3) Preprocess

In [65]:
df = clean_column_names(df)
df = basic_log_preprocess(df, timestamp_col=timestamp_col)
df = add_text_length_features(df, text_cols)
df = fill_numeric(df)
df, encoders = encode_categoricals(df)

# لو عندك label خليه خارج feature columns
feature_cols = [c for c in feature_candidates if c in df.columns]
if not feature_cols:
    feature_cols = [
        c for c in df.columns
        if c != timestamp_col
        and c != label_col
        and pd.api.types.is_numeric_dtype(df[c])
    ]

df = keep_or_create_columns(df, feature_cols)

# مهم: ترتيب زمني قبل sequence
if timestamp_col in df.columns:
    df = df.sort_values(by=timestamp_col).reset_index(drop=True)

X, scaler = scale_features(df, feature_cols)

y = df[label_col].values

X_seq = create_sequences(X, sequence_length=sequence_length)
y_seq = y[sequence_length - 1:]
# split زمني
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq,
    y_seq,
    test_size=0.2,
    random_state=42,
    stratify=y_seq
)
print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("y_train distribution:")
print(pd.Series(y_train).value_counts())

print("y_test distribution:")
print(pd.Series(y_test).value_counts())

print('Rows:', len(df))
print('Features:', len(feature_cols))
print('All sequences:', X_seq.shape)
print('Train sequences:', X_train.shape)
print('Test sequences:', X_test.shape)

X_seq shape: (61056, 10, 18)
y_seq shape: (61056,)
X_train shape: (48844, 10, 18)
X_test shape : (12212, 10, 18)
y_train distribution:
0    28792
1    20052
Name: count, dtype: int64
y_test distribution:
0    7199
1    5013
Name: count, dtype: int64
Rows: 61065
Features: 18
All sequences: (61056, 10, 18)
Train sequences: (48844, 10, 18)
Test sequences: (12212, 10, 18)


In [66]:
print("len(X_seq):", len(X_seq))
print("len(y_seq):", len(y_seq))

len(X_seq): 61056
len(y_seq): 61056


## 4) Build model

In [67]:

n_features = X_seq.shape[2]
inputs = Input(shape=(sequence_length, n_features))
x = LSTM(lstm_units, activation='tanh', return_sequences=True)(inputs)
x = Dropout(dropout_rate)(x)
x = LSTM(latent_dim, activation='tanh', return_sequences=False)(x)
x = RepeatVector(sequence_length)(x)
x = LSTM(latent_dim, activation='tanh', return_sequences=True)(x)
x = Dropout(dropout_rate)(x)
x = LSTM(lstm_units, activation='tanh', return_sequences=True)(x)
outputs = TimeDistributed(Dense(n_features))(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
model.summary()


Model: "model_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_7 (InputLayer)        [(None, 10, 18)]          0         
                                                                 
 lstm_24 (LSTM)              (None, 10, 128)           75264     
                                                                 
 dropout_12 (Dropout)        (None, 10, 128)           0         
                                                                 
 lstm_25 (LSTM)              (None, 32)                20608     
                                                                 
 repeat_vector_6 (RepeatVect  (None, 10, 32)           0         
 or)                                                             
                                                                 
 lstm_26 (LSTM)              (None, 10, 32)            8320      
                                                           

## 5) Train

In [68]:
history = model.fit(
    X_train, X_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, X_test),
    shuffle=False
)

Epoch 1/20
1527/1527 [==============================] - 65s 37ms/step - loss: 0.3201 - val_loss: 0.1486
Epoch 2/20
1527/1527 [==============================] - 61s 40ms/step - loss: 0.1419 - val_loss: 0.1027
Epoch 3/20
1527/1527 [==============================] - 59s 38ms/step - loss: 0.1045 - val_loss: 0.0629
Epoch 4/20
1527/1527 [==============================] - 58s 38ms/step - loss: 0.0679 - val_loss: 0.0443
Epoch 5/20
1527/1527 [==============================] - 55s 36ms/step - loss: 0.0558 - val_loss: 0.0360
Epoch 6/20
1527/1527 [==============================] - 48s 31ms/step - loss: 0.0478 - val_loss: 0.0331
Epoch 7/20
1527/1527 [==============================] - 47s 31ms/step - loss: 0.0411 - val_loss: 0.0272
Epoch 8/20
1527/1527 [==============================] - 47s 31ms/step - loss: 0.0369 - val_loss: 0.0270
Epoch 9/20
1527/1527 [==============================] - 47s 31ms/step - loss: 0.0409 - val_loss: 0.0239
Epoch 10/20
1527/1527 [==============================] - 47s 31m

In [69]:
train_recon = model.predict(X_train, verbose=0)
test_recon = model.predict(X_test, verbose=0)

train_mse = np.mean(np.power(X_train - train_recon, 2), axis=(1,2))
test_mse = np.mean(np.power(X_test - test_recon, 2), axis=(1,2))

print("Train reconstruction error mean:", train_mse.mean())
print("Test reconstruction error mean:", test_mse.mean())
print("Train reconstruction error std:", train_mse.std())
print("Test reconstruction error std:", test_mse.std())

Train reconstruction error mean: 0.01756507403131096
Test reconstruction error mean: 0.01836818032596976
Train reconstruction error std: 0.09556147162122276
Test reconstruction error std: 0.12383397892783905


In [71]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# threshold من train فقط
threshold = float(np.percentile(train_mse, threshold_percentile))
print("Threshold:", threshold)

train_pred_labels = (train_mse > threshold).astype(int)
test_pred_labels = (test_mse > threshold).astype(int)

if y_seq is not None:
    print("\n=== Train Metrics ===")
    print("Accuracy :", accuracy_score(y_train, train_pred_labels))
    print("Precision:", precision_score(y_train, train_pred_labels, zero_division=0))
    print("Recall   :", recall_score(y_train, train_pred_labels, zero_division=0))
    print("F1-score :", f1_score(y_train, train_pred_labels, zero_division=0))

    print("\n=== Test Metrics ===")
    print("Accuracy :", accuracy_score(y_test, test_pred_labels))
    print("Precision:", precision_score(y_test, test_pred_labels, zero_division=0))
    print("Recall   :", recall_score(y_test, test_pred_labels, zero_division=0))
    print("F1-score :", f1_score(y_test, test_pred_labels, zero_division=0))

    print("\n=== Classification Report (Test) ===")
    print(classification_report(y_test, test_pred_labels, zero_division=0))

    print("\n=== Confusion Matrix (Test) ===")
    print(confusion_matrix(y_test, test_pred_labels))
else:
    print("No label column found -> accuracy cannot be computed.")

Threshold: 0.006936572690702766

=== Train Metrics ===
Accuracy : 0.9392760625665384
Precision: 0.9997075339260646
Recall   : 0.8523339317773788
F1-score : 0.9201572090018305

=== Test Metrics ===
Accuracy : 0.9337536849000982
Precision: 0.9997622444127438
Recall   : 0.8388190704169161
F1-score : 0.9122464475539647

=== Classification Report (Test) ===
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      7199
           1       1.00      0.84      0.91      5013

    accuracy                           0.93     12212
   macro avg       0.95      0.92      0.93     12212
weighted avg       0.94      0.93      0.93     12212


=== Confusion Matrix (Test) ===
[[7198    1]
 [ 808 4205]]


In [72]:
train_anomalies = (train_mse > threshold).sum()
test_anomalies = (test_mse > threshold).sum()

print("Train anomalies:", train_anomalies, "out of", len(train_mse))
print("Test anomalies :", test_anomalies, "out of", len(test_mse))

Train anomalies: 17096 out of 48844
Test anomalies : 4206 out of 12212


## 6) Compute threshold and save artifacts

In [2]:
threshold = float(np.percentile(train_mse, threshold_percentile))
print('Threshold:', threshold)

model.save(os.path.join(MODEL_DIR, 'web_log_lstm_autoencoder.keras'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'web_log_scaler.joblib'))

config = {
    'source': 'web',
    'timestamp_col': timestamp_col,
    'text_cols': text_cols,
    'feature_cols': feature_cols,
    'sequence_length': sequence_length,
    'lstm_units': lstm_units,
    'latent_dim': latent_dim,
    'dropout_rate': dropout_rate,
    'learning_rate': learning_rate,
    'batch_size': batch_size,
    'epochs': epochs,
    'threshold_percentile': threshold_percentile,
    'threshold': threshold,
    'train_ratio': train_ratio,
    'label_col': label_col,
}
with open(os.path.join(MODEL_DIR, 'web_log_model_config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
model.save(os.path.join(MODEL_DIR, 'web_log_lstm_autoencoder.h5'))

print('Saved to', MODEL_DIR)

NameError: name 'np' is not defined